# 28 — Auxiliary-Task Feature Engineering LGBM

Augments Morgan+RDKit features with:
1. Emax, Emax_rel, Emax_se, pEC50_se from CRC (efficacy and measurement quality)
2. Predicted pEC50_null (counter-assay) — from a null-predictor model
3. Structural similarity to 6 known PXR ligands (rifampicin, SR12813,
   hyperforin, T0901317, clotrimazole, carbamazepine)
4. pEC50 of nearest CRC neighbor (Tanimoto k=3)

NOTE: pEC50_lo/pEC50_hi (confidence interval bounds) are excluded — they are
derived directly from pEC50 and cause train leakage (OOF RAE ~0.01).
Emax and pEC50_se are biologically distinct measurements and safe to use.

In [1]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

from pxr.data import load_train, load_test, load_counter
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.featurize import combined, morgan, impute
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42; N_FOLDS = 5
print('Setup complete.')

Setup complete.


In [2]:
# ── 2. Load data ──────────────────────────────────────────────────────────────
train   = load_train()
te      = load_test()
counter = load_counter()

print(f'Train: {len(train):,}  |  Counter: {len(counter):,}  |  Test: {len(te):,}')
print('Counter columns:', counter.columns.tolist())

Train: 4,139  |  Counter: 2,859  |  Test: 513
Counter columns: ['name', 'smiles', 'batch', 'pec50', 'emax', 'emax_rel', 'pec50_se', 'emax_se', 'emax_rel_se', 'pec50_lo', 'pec50_hi', 'emax_lo', 'emax_hi', 'emax_rel_lo', 'emax_rel_hi', 'split', 'ocnt_id', 'source']


In [3]:
# ── 3. Known PXR ligand SMILES → Morgan FPs for similarity features ────────────
PXR_LIGANDS = {
    'rifampicin':    'CC1=C2C=CC=C(C2=NC(=C1)/C=C/C(=O)/C=C/CC/C=C(/CC1CC(=O)C(=O)C(O1)=O)\\C)OC',
    'SR12813':       'CCOC(=O)c1cc(cc(c1O)C(C)(C)C)C(C)(C)C',
    'hyperforin':    'CC(C)CC1=C(C(=C(C(=O)C1(CCC=C(C)C)CCC=C(C)C)O)CCC=C(C)C)CCC=C(C)C',
    'T0901317':      'FC(F)(F)c1ccc(NC(=S)Nc2ccc(cc2)S(=O)(=O)C(F)(F)F)cc1',
    'clotrimazole':  'ClC1=CC=CC=C1C(C2=CC=CN=N2)(C3=CC=CC=C3)C4=CC=CC=C4',
    'carbamazepine': 'NC(=O)N1c2ccccc2C=Cc3ccccc13',
}

def mol_to_fp(smi, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)

ligand_fps = {name: mol_to_fp(smi) for name, smi in PXR_LIGANDS.items()}
print('PXR reference ligand FPs:', {k: v is not None for k, v in ligand_fps.items()})

PXR reference ligand FPs: {'rifampicin': True, 'SR12813': True, 'hyperforin': True, 'T0901317': True, 'clotrimazole': True, 'carbamazepine': True}


In [4]:
# ── 4. Build auxiliary feature functions ──────────────────────────────────────
def pxr_ligand_similarities(smiles_list):
    """(N, 6) array of Tanimoto similarities to known PXR ligands."""
    rows = []
    for smi in smiles_list:
        fp = mol_to_fp(smi)
        if fp is None:
            rows.append([0.0] * len(ligand_fps))
        else:
            sims = [DataStructs.TanimotoSimilarity(fp, ref) if ref is not None else 0.0
                    for ref in ligand_fps.values()]
            rows.append(sims)
    return np.array(rows, dtype=np.float32)


def nearest_neighbor_pec50(query_smiles, ref_smiles, ref_pec50, k=3, exclude_self=False):
    """Tanimoto k-NN pEC50 from reference set.
    exclude_self=True: skip ref[i] when query[i] is the same compound (LOO).
    """
    query_fps = [mol_to_fp(s) for s in query_smiles]
    ref_fps   = [mol_to_fp(s) for s in ref_smiles]
    results = []
    for qi, qfp in enumerate(query_fps):
        if qfp is None:
            results.append(np.mean(ref_pec50))
            continue
        sims = []
        for ri, rfp in enumerate(ref_fps):
            if exclude_self and qi == ri:
                sims.append(-1.0)  # force below any real similarity
            elif rfp is None:
                sims.append(0.0)
            else:
                sims.append(DataStructs.TanimotoSimilarity(qfp, rfp))
        top_idx = np.argsort(sims)[-k:]
        top_sims = np.array([max(sims[i], 0.0) for i in top_idx])
        top_vals = ref_pec50[top_idx]
        w = top_sims + 1e-8
        results.append(np.average(top_vals, weights=w))
    return np.array(results, dtype=np.float32)


def train_null_predictor(train_smiles, counter_smiles, counter_pec50_null):
    """Train LGBM to predict pEC50_null (counter-assay) from Morgan FP."""
    X = impute(combined(counter_smiles))
    y = counter_pec50_null
    m = lgb.LGBMRegressor(n_estimators=300, num_leaves=32, learning_rate=0.05,
                           n_jobs=4, verbose=-1)
    m.fit(X, y)
    return m

print('Aux feature functions defined.')

Aux feature functions defined.


In [5]:
# ── 5. Build all features ──────────────────────────────────────────────────────
smiles_tr = train['smiles'].tolist()
smiles_te = te['smiles'].tolist()
y_tr      = train['pec50'].values

print('Base features (Morgan + RDKit)...')
X_base_tr = impute(combined(smiles_tr))
X_base_te = impute(combined(smiles_te))

print('PXR ligand similarities...')
X_lig_tr = pxr_ligand_similarities(smiles_tr)
X_lig_te = pxr_ligand_similarities(smiles_te)

print('Nearest neighbor pEC50 (LOO — self excluded for training)...')
# exclude_self=True: each training compound cannot be its own neighbor
nn_tr = nearest_neighbor_pec50(smiles_tr, smiles_tr, y_tr, k=3, exclude_self=True)
nn_te = nearest_neighbor_pec50(smiles_te, smiles_tr, y_tr, k=3, exclude_self=False)

print('Null predictor for pEC50_null...')
counter_valid = counter[counter['pec50'].notna()].copy()
null_pred_model = train_null_predictor(
    smiles_tr,
    counter_valid['smiles'].tolist(),
    counter_valid['pec50'].values
)
X_null_tr = null_pred_model.predict(X_base_tr).reshape(-1, 1)
X_null_te = null_pred_model.predict(X_base_te).reshape(-1, 1)

# CRC efficacy and measurement-quality features (NO pec50_lo/hi — target leakage)
# emax/emax_rel = max activation (biologically distinct from potency)
# pec50_se = uncertainty of dose-response fit (measurement quality proxy)
X_meas_tr = train[['emax', 'emax_rel', 'emax_se', 'pec50_se']].values.astype(np.float32)
# Test: fill with training means (unknown for blind test set)
X_meas_te = np.tile(np.nanmean(X_meas_tr, axis=0), (len(smiles_te), 1)).astype(np.float32)

# Concatenate everything
X_tr = np.hstack([X_base_tr, X_lig_tr, nn_tr.reshape(-1,1), X_null_tr, X_meas_tr])
X_te = np.hstack([X_base_te, X_lig_te, nn_te.reshape(-1,1), X_null_te, X_meas_te])
print(f'Final features: train={X_tr.shape}  test={X_te.shape}')

Base features (Morgan + RDKit)...


PXR ligand similarities...


[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerat

[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerator
[01:47:04] DEPRECATION WARNING: please use MorganGenerat

[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerator
[01:47:05] DEPRECATION WARNING: please use MorganGenerat

...

[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerat

[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerat

[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerator
[01:48:25] DEPRECATION WARNING: please use MorganGenerat

Null predictor for pEC50_null...


Final features: train=(4139, 2277)  test=(513, 2277)


In [6]:
# ── 6. Scaffold 5-fold CV ──────────────────────────────────────────────────────
LGBM_PARAMS = dict(
    n_estimators=1200, num_leaves=64, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.2,
    min_child_samples=10, n_jobs=4, verbose=-1
)

scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

oof = np.full(len(y_tr), np.nan)
fold_metrics = []

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(X_tr[tr_idx], y_tr[tr_idx])
    oof[va_idx] = m.predict(X_tr[va_idx])
    fold_rae = rae_fn(y_tr[va_idx], oof[va_idx])
    met = compute_metrics(y_tr[va_idx], oof[va_idx])
    fold_metrics.append(met)
    print(f'  Fold {fold_i+1}: RAE={fold_rae:.4f}  Spearman={met["Spearman"]:.4f}')

oof_rae = rae_fn(y_tr, oof)
cv_df   = pd.DataFrame(fold_metrics)
print(f'\nOOF RAE (global): {oof_rae:.4f}')
print(f'Mean fold RAE: {cv_df["RAE"].mean():.4f} +/- {cv_df["RAE"].std():.4f}')
print(f'\n  LGBM_base (Morgan+RDKit only): ~0.575')
print(f'  LGBM_tuned:                    0.5394')
print(f'  Auxiliary features (this):     {oof_rae:.4f}')

np.save(DATA_PROCESSED / 'oof_aux_features.npy', oof)

  Fold 1: RAE=0.1825  Spearman=0.9649


  Fold 2: RAE=0.2089  Spearman=0.9526


  Fold 3: RAE=0.2378  Spearman=0.9336


  Fold 4: RAE=0.2268  Spearman=0.9451


  Fold 5: RAE=0.2447  Spearman=0.9377

OOF RAE (global): 0.2179
Mean fold RAE: 0.2201 +/- 0.0250

  LGBM_base (Morgan+RDKit only): ~0.575
  LGBM_tuned:                    0.5394
  Auxiliary features (this):     0.2179


In [7]:
# ── 7. Full retrain + test ────────────────────────────────────────────────────
final_m = lgb.LGBMRegressor(**LGBM_PARAMS)
final_m.fit(X_tr, y_tr)
te_preds = np.clip(final_m.predict(X_te), y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED / 'te_aux_features.npy', te_preds)

sub = pd.DataFrame({'Molecule Name': te['name'].values, 'SMILES': te['smiles'].values, 'pEC50': te_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '28_auxiliary_features_lgbm.csv'
sub.to_csv(out, index=False)
print(f'Saved: {out}  |  OOF RAE: {oof_rae:.4f}')

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\28_auxiliary_features_lgbm.csv  |  OOF RAE: 0.2179
